# Lesson 06-2：SQL 基礎查詢

示範如何用 SQLite 搭配 pandas 執行 SQL 查詢，完成訂單、顧客與商品的多表分析。

學習目標：
- 連線到本課使用的 SQLite 資料庫
- 使用 SQL 彙總完成訂單營收
- 用 `JOIN` 連接訂單、商品與顧客資料
- 依城市、品類、付款方式與顧客做分析


## 1. 載入套件與建立連線

`sqlite3.connect()` 會連到 `data/raw/course.db`。後續使用 `pd.read_sql_query(sql, conn)`，就能把 SQL 查詢結果轉成 pandas DataFrame。


In [1]:
import sqlite3
import pandas as pd
from common import RAW, ensure_packages

ensure_packages()

db_path = RAW / "course.db"
conn = sqlite3.connect(db_path)

print(db_path)


E:\py_20260620\data\raw\course.db


## 2. 查看資料庫中的資料表

SQLite 會把資料庫結構資訊放在 `sqlite_master`。先確認有哪些資料表，有助於後面寫 `JOIN` 時選對表格。


In [ ]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    conn,
)

tables


,name
0,ab_assignments
1,customers
2,events
3,order_items
4,orders
5,products
6,sessions


## 3. 依付款方式分析平均營收

先在子查詢中把 `order_items` 彙總成每張訂單的營收，再和 `orders` 合併，最後只分析完成訂單。

SQL 重點：
- 子查詢：先計算每張訂單的 `revenue`
- `JOIN`：把訂單營收接回訂單資料
- `WHERE`：只保留 `completed`
- `GROUP BY`：依付款方式彙總


In [ ]:
pd.read_sql_query('''
    SELECT
        order_id,
        SUM(
            CAST(quantity AS REAL)
            * CAST(unit_price AS REAL)
            * (1 - CAST(discount_rate AS REAL))
        ) AS revenue
    FROM order_items
    GROUP BY order_id
''', conn)

,order_id,revenue
0,1,538.65
1,2,4357.70
2,3,3536.00
3,4,1193.40
4,5,13175.50
...,...,...
21995,21996,6198.25
21996,21997,3160.00
21997,21998,951.90
21998,21999,2204.00


In [3]:
payment_query = """
SELECT
    o.payment_type,
    COUNT(*) AS orders,
    ROUND(AVG(CAST(i.revenue AS REAL)), 2) AS avg_revenue
FROM (
    SELECT
        order_id,
        SUM(
            CAST(quantity AS REAL)
            * CAST(unit_price AS REAL)
            * (1 - CAST(discount_rate AS REAL))
        ) AS revenue
    FROM order_items
    GROUP BY order_id
) i
JOIN orders o ON o.order_id = i.order_id
WHERE o.status = 'completed'
GROUP BY o.payment_type
ORDER BY avg_revenue DESC;
"""

payment_result = pd.read_sql_query(payment_query, conn)
payment_result


,payment_type,orders,avg_revenue
0,cod,2060,5052.66
1,wallet,3076,4987.98
2,atm,5111,4979.96
3,card,10166,4974.20


## 4. 練習 1：各城市 VIP 顧客數

查詢 `customers` 表，篩選 `segment = 'vip'`，再依城市統計人數。


In [9]:
vip_city_query = """
SELECT
    city,
    COUNT(*) AS vip_count
FROM customers
WHERE segment = 'vip'
GROUP BY city
ORDER BY vip_count DESC;
"""

vip_by_city = pd.read_sql_query(vip_city_query, conn)
vip_by_city


,city,vip_count
0,Tainan,56
1,Hsinchu,46
2,Kaohsiung,41
3,Taichung,40
4,Taoyuan,37
5,Taipei,36


## 5. 練習 2：依商品品類計算營收

這題需要三張表：

- `order_items`：數量、單價、折扣率
- `products`：商品品類
- `orders`：訂單狀態

只計算完成訂單，避免把取消或未完成訂單算進營收。


In [10]:
category_revenue_query = """
SELECT
    p.category,
    ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_rate)), 2) AS total_revenue
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN orders o ON oi.order_id = o.order_id
WHERE o.status = 'completed'
GROUP BY p.category
ORDER BY total_revenue DESC;
"""

revenue_by_category = pd.read_sql_query(category_revenue_query, conn)
revenue_by_category


,category,total_revenue
0,beauty,18283494.35
1,home,18061948.35
2,sports,17428979.90
3,electronics,17368081.05
4,grocery,16824601.25
5,fashion,13804694.85


## 6. 練習 3：找出消費最高的前 10 位顧客

這題把 `orders`、`order_items`、`customers` 三張表串起來，計算每位顧客在完成訂單中的總消費金額。

`LIMIT 10` 會只保留排序後的前 10 筆，適合用在排行榜類型的查詢。


In [11]:
top_customers_query = """
SELECT
    o.customer_id,
    c.segment,
    c.city,
    ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_rate)), 2) AS total_spent
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.status = 'completed'
GROUP BY o.customer_id, c.segment, c.city
ORDER BY total_spent DESC
LIMIT 10;
"""

top_customers = pd.read_sql_query(top_customers_query, conn)
top_customers


,customer_id,segment,city,total_spent
0,541,vip,Hsinchu,106597.85
1,1717,growth,Tainan,100774.50
2,2079,new,Tainan,100759.10
3,1483,new,Tainan,100045.75
4,53,vip,Taoyuan,100029.40
5,1729,growth,Taichung,99322.95
6,18,growth,Taichung,98810.60
7,1602,new,Kaohsiung,98540.70
8,1705,growth,Hsinchu,96237.65
9,2275,growth,Hsinchu,95695.25


## 7. 查詢結果解讀

觀察：

- 排名最高的付款方式是否只是平均高，還是訂單數也夠多？
- VIP 顧客是否集中在少數城市？
- 哪些商品品類貢獻最多完成訂單營收？
- Top 顧客來自哪些客群與城市？

這些問題能幫助我們把表格結果轉成商業判斷。


## 8. 小練習

請建立一個查詢，依 `customers.segment` 分組，計算完成訂單的：

- `orders`：訂單數
- `customers`：不重複顧客數
- `total_revenue`：總營收
- `avg_revenue`：平均訂單營收

提示：可以從 `orders` 接 `order_items` 與 `customers`，再用 `GROUP BY c.segment`。


In [ ]:
segment_sql = """
SELECT
    c.segment,
    COUNT(DISTINCT o.order_id) AS orders,
    COUNT(DISTINCT o.customer_id) AS customers,
    ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_rate)), 2) AS total_revenue,
    ROUND(
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_rate))
        / COUNT(DISTINCT o.order_id),
        2
    ) AS avg_revenue
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.status = 'completed'
GROUP BY c.segment
ORDER BY total_revenue DESC;
"""

segment_result = pd.read_sql_query(segment_sql, conn)
segment_result


## 9. 常見錯誤與延伸

常見錯誤：
- 忘記加上 `WHERE o.status = 'completed'`，導致未完成訂單也被算進營收。
- 在多表 `JOIN` 時沒有確認 key，例如 `order_id`、`product_id`、`customer_id` 是否接對。
- 用 `COUNT(*)` 計算訂單數時，如果查詢仍在明細層級，可能算到商品列數而不是訂單數。

延伸練習：
- 把第 8 節的小練習加上 `payment_type`，做成「客群 x 付款方式」分析。
- 將 SQL 結果存成 DataFrame 後，用 pandas 再計算營收占比。


## 10. 關閉資料庫連線

分析完成後關閉連線，釋放資料庫資源。


In [12]:
conn.close()
print("資料庫連線已關閉。")


資料庫連線已關閉。
